In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/name_mapping.csv
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/survival_data.csv
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_seg.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_t1.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_t1ce.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_flair.nii
/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training/HGG/BraTS19_2013_27_1/BraTS19_2013_27_1_t2.nii
/kaggle/in

In [5]:
!pip install nibabel

In [6]:
import os
import random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

DATA_ROOT = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
SAVE_DIR  = "/kaggle/working/outputs1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4
EPOCHS = 80
LR = 1e-4
NUM_CASES  = 80
NUM_SLICES = 40

os.makedirs(SAVE_DIR, exist_ok=True)

# ================= UTILS =================

def normalize(img):
    return (img - img.mean()) / (img.std() + 1e-8)

def center_crop(img, size=192):
    h, w = img.shape
    s = size // 2
    return img[h//2-s:h//2+s, w//2-s:w//2+s]

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/mse)

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())

# ================= DATASET =================

class BraTSDataset(Dataset):
    def __init__(self, root):
        self.data = []
        all_cases = []

        for folder in ["HGG","LGG"]:
            path=os.path.join(root,folder)
            if os.path.exists(path):
                all_cases += [os.path.join(path,c) for c in os.listdir(path)]

        all_cases=all_cases[:NUM_CASES]

        for case in all_cases:
            files=os.listdir(case)

            t1=t2=t1ce=flair=None
            for f in files:
                name=f.lower()
                if "_t1." in name and "ce" not in name: t1=f
                elif "_t2." in name: t2=f
                elif "t1ce" in name: t1ce=f
                elif "flair" in name: flair=f

            if None in [t1,t2,t1ce,flair]: continue

            vols=[]
            for fname in [t1,t2,t1ce,flair]:
                vol=nib.load(os.path.join(case,fname)).get_fdata(dtype=np.float32)
                z=vol.shape[2]
                start=(z-NUM_SLICES)//2
                vols.append(vol[:,:,start:start+NUM_SLICES])

            self.data.append(vols)

    def __len__(self):
        return len(self.data)*NUM_SLICES

    def __getitem__(self, idx):
        case_id = idx//NUM_SLICES
        slice_id = idx%NUM_SLICES

        vols=self.data[case_id]

        imgs=[]
        gt_imgs=[]
        for vol in vols:
            img=center_crop(vol[:,:,slice_id])
            imgs.append(normalize(img))
            gt_imgs.append(normalize(img))

        imgs = np.stack(imgs)
        gt_imgs = np.stack(gt_imgs)

        imgs=torch.from_numpy(imgs).float().unsqueeze(1)
        gt = torch.from_numpy(gt_imgs).float().unsqueeze(1)

        ac=torch.ones(4)
        miss=1
        idxs=random.sample(range(4),miss)
        for i in idxs:
            ac[i]=0
            imgs[i]=0

        return imgs,ac,gt

# ================= MODEL =================

class ConvBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1),
            nn.InstanceNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.block(x)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks=nn.ModuleList([
            ConvBlock(1,16),
            ConvBlock(16,32),
            ConvBlock(32,64)
        ])
        self.pool=nn.MaxPool2d(2)

    def forward(self,x):
        feats=[]
        for b in self.blocks:
            x=b(x)
            feats.append(x)
            x=self.pool(x)
        return feats

class DFUM(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.att3=nn.Conv2d(c,c,3,1,1)
        self.att5=nn.Conv2d(c,c,5,1,2)
        self.att7=nn.Conv2d(c,c,7,1,3)
        self.fuse=nn.Conv2d(2*c,c,1)

    def forward(self,feats,ac):
        mask=ac.view(-1,1,1,1)
        valid=[feats[i]*mask[i] for i in range(len(feats))]
        valid=torch.stack(valid)

        hard=torch.max(valid,dim=0)[0]

        att=[self.att3(f)+self.att5(f)+self.att7(f) for f in valid]
        w=torch.softmax(torch.stack(att),dim=0)

        soft=sum(wi*fi for wi,fi in zip(w,valid))
        return self.fuse(torch.cat([hard,soft],1))

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2)
        self.blocks=nn.ModuleList([
            ConvBlock(64,32),
            ConvBlock(32,16)
        ])
        self.out=nn.Conv2d(16,1,1)

    def forward(self,feats):
        x=feats[-1]
        for b in self.blocks:
            x=self.up(x)
            x=b(x)
        return self.out(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.ModuleList([Encoder() for _ in range(4)])
        self.dfums=nn.ModuleList([DFUM(16),DFUM(32),DFUM(64)])
        self.dec=nn.ModuleList([Decoder() for _ in range(4)])

    def forward(self,x,ac):
        spec=[self.enc[i](x[i]) for i in range(4)]
        unified=[]
        for s in range(3):
            feats=[spec[i][s] for i in range(4)]
            unified.append(self.dfums[s](feats,ac))
        outputs=[d(unified) for d in self.dec]
        return torch.tanh(torch.stack(outputs))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1),
            nn.InstanceNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,1)
        )
    def forward(self,x): return self.net(x)

MSE=nn.MSELoss()
def g_loss(fake): return MSE(fake,torch.ones_like(fake))
def d_loss(real,fake): return 0.5*(MSE(real,torch.ones_like(real))+MSE(fake,torch.zeros_like(fake)))

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/(mse+1e-8))

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())

# ================= IMAGE SAVE =================

def save_sample(imgs, fake, ac, epoch):
    imgs = imgs.cpu()
    fake = fake.cpu()
    ac   = ac.cpu()
    names = ['T1','T2','T1CE','FLAIR']

    for i in range(4):
        inp = imgs[i,0,0].numpy()
        syn = fake[i,0,0].detach().numpy()
        status = "Missing" if ac[i]==0 else "Present"

        fig,ax=plt.subplots(1,2,figsize=(6,3))
        ax[0].imshow(inp,cmap="gray")
        ax[0].set_title(f"{names[i]} Input ({status})")
        ax[0].axis("off")
        ax[1].imshow(syn,cmap="gray")
        ax[1].set_title(f"{names[i]} Synth")
        ax[1].axis("off")
        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/epoch{epoch}_{names[i]}.png")
        plt.close()

def gradient_loss(pred, target):

    def grad(x):
        gx = x[:,:,:,1:] - x[:,:,:,:-1]
        gy = x[:,:,1:,:] - x[:,:,:-1,:]
        return gx, gy

    gx1, gy1 = grad(pred)
    gx2, gy2 = grad(target)

    return L1(gx1, gx2) + L1(gy1, gy2)


# ================= TRAIN =================

dataset=BraTSDataset(DATA_ROOT)
loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True)

netG=Generator().to(DEVICE)
netD=[Discriminator().to(DEVICE) for _ in range(4)]

optG=torch.optim.Adam(netG.parameters(),lr=LR,betas=(0.5,0.999))
optD=[torch.optim.Adam(d.parameters(),lr=LR,betas=(0.5,0.999)) for d in netD]
L1=nn.L1Loss()

for epoch in range(EPOCHS):

    netG.train()

    for imgs,ac,gt in loader:

        imgs=imgs.to(DEVICE)
        ac=ac.to(DEVICE)

        if imgs.dim()==4:
            imgs = imgs.unsqueeze(0)

        imgs = imgs.permute(1,0,2,3,4)
        gt = gt.to(DEVICE)
        if gt.dim()==4:
            gt = gt.unsqueeze(0)

        gt = gt.permute(1,0,2,3,4)

        fake = netG(imgs, ac[0])

        loss_syn=sum((1-ac[0,i])*L1(fake[i],gt[i]) for i in range(4))
        loss_rec=sum(ac[0,i]*L1(fake[i],gt[i]) for i in range(4))

        adv=0
        grad = 0
        for i in range(4):
            if ac[0,i]==0:
                adv+=g_loss(netD[i](fake[i]))
                grad += gradient_loss(fake[i], gt[i])

                
        lossG = 70*loss_syn + 20*loss_rec + 10*adv + 5*grad

        optG.zero_grad()
        lossG.backward()
        optG.step()

        for i in range(4):
            if ac[0,i]==0:
                optD[i].zero_grad()
                lossD=d_loss(netD[i](gt[i]), netD[i](fake[i].detach()))

                lossD.backward()
                optD[i].step()

        # ======== EVALUATION ========
    netG.eval()
    psnr_vals=[]
    ssim_vals=[]

    with torch.no_grad():

        for imgs,ac,gt in loader:

            imgs=imgs.to(DEVICE)
            ac=ac.to(DEVICE)

            imgs=imgs.permute(1,0,2,3,4)
            fake = netG(imgs, ac[0])

            for i in range(4):

                if ac[0,i]==0:

                    pred=fake[i][0,0].cpu().numpy()
                    gt=gt[i][0,0].cpu().numpy()

                    psnr_vals.append(compute_psnr(pred,gt))
                    ssim_vals.append(compute_ssim(pred,gt))

            save_sample(imgs ,fake,ac[0],epoch)
            break

    print(f"Epoch {epoch} | PSNR {np.mean(psnr_vals):.2f} | SSIM {np.mean(ssim_vals):.4f}")



Epoch 0 | PSNR 7.43 | SSIM 0.5647
Epoch 1 | PSNR 6.83 | SSIM 0.5461
Epoch 2 | PSNR 11.72 | SSIM 0.7359
Epoch 3 | PSNR 12.54 | SSIM 0.7007
Epoch 4 | PSNR 11.65 | SSIM 0.6692
Epoch 5 | PSNR 4.19 | SSIM 0.3621
Epoch 6 | PSNR 5.78 | SSIM 0.4598
Epoch 7 | PSNR 5.58 | SSIM 0.4827
Epoch 8 | PSNR 5.36 | SSIM 0.4005
Epoch 9 | PSNR 11.83 | SSIM 0.6755
Epoch 10 | PSNR 5.85 | SSIM 0.4232
Epoch 11 | PSNR 13.52 | SSIM 0.7035
Epoch 12 | PSNR 5.35 | SSIM 0.3993
Epoch 13 | PSNR 11.70 | SSIM 0.5953
Epoch 14 | PSNR 7.24 | SSIM 0.3610
Epoch 15 | PSNR 3.24 | SSIM 0.3562
Epoch 16 | PSNR 12.15 | SSIM 0.6348
Epoch 17 | PSNR 11.15 | SSIM 0.6125
Epoch 18 | PSNR 6.46 | SSIM 0.2477
Epoch 19 | PSNR 4.23 | SSIM 0.3961
Epoch 20 | PSNR 4.34 | SSIM 0.3291
Epoch 21 | PSNR 5.40 | SSIM 0.4430
Epoch 22 | PSNR 4.67 | SSIM 0.3250
Epoch 23 | PSNR 4.94 | SSIM 0.4264
Epoch 24 | PSNR 4.00 | SSIM 0.3428
Epoch 25 | PSNR 11.89 | SSIM 0.5920
Epoch 26 | PSNR 3.65 | SSIM 0.3751
Epoch 27 | PSNR 11.55 | SSIM 0.6304
Epoch 28 | PSNR 5.34

KeyboardInterrupt: 

In [2]:
!pip install antspyx
!pip install torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 74.2 MB/s eta 0:00:00:00:0100:01


In [3]:
import os
import random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt
import ants
import torchvision.models as models

DATA_ROOT = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
SAVE_DIR  = "/kaggle/working/outputs"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 2
EPOCHS = 60
LR = 1e-4
NUM_CASES  = 40
NUM_SLICES = 32

os.makedirs(SAVE_DIR, exist_ok=True)

def normalize(img):
    img = img - np.min(img)
    img = img / (np.max(img) + 1e-8)
    return img

def center_crop(img, size=192):
    h, w = img.shape
    s = size // 2
    return img[h//2-s:h//2+s, w//2-s:w//2+s]

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/(mse+1e-8))

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())

def register_to_flair(moving, fixed):

    moving_img = ants.from_numpy(moving)
    fixed_img  = ants.from_numpy(fixed)

    tx = ants.registration(fixed=fixed_img, moving=moving_img, type_of_transform="Affine")

    warped = ants.apply_transforms(
        fixed=fixed_img,
        moving=moving_img,
        transformlist=tx['fwdtransforms']
    )

    return warped.numpy()

class BraTSDataset(Dataset):
    def __init__(self, root):

        self.data=[]
        all_cases=[]

        for folder in ["HGG","LGG"]:
            path=os.path.join(root,folder)
            if os.path.exists(path):
                all_cases += [os.path.join(path,c) for c in os.listdir(path)]

        all_cases=all_cases[:NUM_CASES]

        print("Registering modalities...")

        for case in all_cases:

            files=os.listdir(case)

            t1=t2=t1ce=flair=None
            for f in files:
                name=f.lower()
                if "_t1." in name and "ce" not in name: t1=f
                elif "_t2." in name: t2=f
                elif "t1ce" in name: t1ce=f
                elif "flair" in name: flair=f

            if None in [t1,t2,t1ce,flair]: continue

            flair_vol=nib.load(os.path.join(case,flair)).get_fdata(dtype=np.float32)

            vols=[]
            for fname in [t1,t2,t1ce]:

                moving=nib.load(os.path.join(case,fname)).get_fdata(dtype=np.float32)
                registered=register_to_flair(moving, flair_vol)

                z=registered.shape[2]
                start=(z-NUM_SLICES)//2
                vols.append(registered[:,:,start:start+NUM_SLICES])

            z=flair_vol.shape[2]
            start=(z-NUM_SLICES)//2
            vols.append(flair_vol[:,:,start:start+NUM_SLICES])

            self.data.append(vols)

    def __len__(self):
        return len(self.data)*NUM_SLICES

    def __getitem__(self, idx):

        case_id=idx//NUM_SLICES
        slice_id=idx%NUM_SLICES

        vols=self.data[case_id]

        imgs=[]
        for vol in vols:
            img=center_crop(vol[:,:,slice_id])
            imgs.append(normalize(img))

        imgs=np.stack(imgs)
        gt=imgs.copy()

        imgs=torch.from_numpy(imgs).float().unsqueeze(1)
        gt=torch.from_numpy(gt).float().unsqueeze(1)

        ac=torch.ones(4)
        miss=random.randint(1,3)
        idxs=random.sample(range(4),miss)
        for i in idxs:
            ac[i]=0
            imgs[i]=0

        return imgs,ac,gt

vgg=models.vgg16(pretrained=True).features[:16].to(DEVICE).eval()
for p in vgg.parameters(): p.requires_grad=False

def perceptual_loss(pred, target):
    pred3 = pred.repeat(1,3,1,1)
    target3 = target.repeat(1,3,1,1)
    return nn.L1Loss()(vgg(pred3), vgg(target3))


class ConvBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.block=nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1),
            nn.InstanceNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self,x): return self.block(x)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks=nn.ModuleList([
            ConvBlock(1,16),
            ConvBlock(16,32),
            ConvBlock(32,64)
        ])
        self.pool=nn.MaxPool2d(2)

    def forward(self,x):
        feats=[]
        for b in self.blocks:
            x=b(x)
            feats.append(x)
            x=self.pool(x)
        return feats

class DFUM(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.att3=nn.Conv2d(c,c,3,1,1)
        self.att5=nn.Conv2d(c,c,5,1,2)
        self.att7=nn.Conv2d(c,c,7,1,3)
        self.fuse=nn.Conv2d(2*c,c,1)

    def forward(self,feats,ac):
        mask=ac.view(-1,1,1,1)
        valid=[feats[i]*mask[i] for i in range(len(feats))]
        valid=torch.stack(valid)

        hard=torch.max(valid,dim=0)[0]

        att=[self.att3(f)+self.att5(f)+self.att7(f) for f in valid]
        w=torch.softmax(torch.stack(att),dim=0)

        soft=sum(wi*fi for wi,fi in zip(w,valid))
        return self.fuse(torch.cat([hard,soft],1))

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.up=nn.Upsample(scale_factor=2)
        self.blocks=nn.ModuleList([
            ConvBlock(64,32),
            ConvBlock(32,16)
        ])
        self.out=nn.Conv2d(16,1,1)

    def forward(self,feats):
        x=feats[-1]
        for b in self.blocks:
            x=self.up(x)
            x=b(x)
        return self.out(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc=nn.ModuleList([Encoder() for _ in range(4)])
        self.dfums=nn.ModuleList([DFUM(16),DFUM(32),DFUM(64)])
        self.dec=nn.ModuleList([Decoder() for _ in range(4)])

    def forward(self,x,ac):
        spec=[self.enc[i](x[i]) for i in range(4)]
        unified=[]
        for s in range(3):
            feats=[spec[i][s] for i in range(4)]
            unified.append(self.dfums[s](feats,ac))
        outputs=[d(unified) for d in self.dec]
        return torch.sigmoid(torch.stack(outputs))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(32,64,4,2,1),
            nn.InstanceNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,128,4,2,1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Conv2d(128,1,4,1,1)
        )
    def forward(self,x): return self.net(x)

# ================= IMAGE SAVE =================

def denormalize(x):
    x = (x + 1) / 2   # convert from [-1,1] → [0,1]
    return np.clip(x, 0, 1)

def enhance_contrast(img):
    p2, p98 = np.percentile(img, (2, 98))
    img = np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)
    return img

def save_sample(imgs, fake, ac, epoch):

    imgs = imgs.cpu()
    fake = fake.cpu()
    ac   = ac.cpu()

    names = ['T1','T2','T1CE','FLAIR']

    for i in range(4):

        inp = imgs[i,0,0].numpy()
        syn = fake[i,0,0].detach().numpy()

        inp = enhance_contrast(denormalize(inp))
        syn = enhance_contrast(denormalize(syn))

        status = "Missing" if ac[i]==0 else "Present"

        fig,ax=plt.subplots(1,2,figsize=(6,3))

        ax[0].imshow(inp, cmap="gray", vmin=0, vmax=1)
        ax[0].set_title(f"{names[i]} Input ({status})")
        ax[0].axis("off")

        ax[1].imshow(syn, cmap="gray", vmin=0, vmax=1)
        ax[1].set_title(f"{names[i]} Synth")
        ax[1].axis("off")

        plt.tight_layout()
        plt.savefig(f"{SAVE_DIR}/epoch{epoch}_{names[i]}.png")
        plt.close()


MSE=nn.MSELoss()
def g_loss(fake): return MSE(fake,torch.ones_like(fake))
def d_loss(real,fake): return 0.5*(MSE(real,torch.ones_like(real))+MSE(fake,torch.zeros_like(fake)))

def compute_psnr(pred, target):
    mse = np.mean((pred-target)**2)
    return 10*np.log10(1.0/(mse+1e-8))

def compute_ssim(pred, target):
    return ssim(pred, target, data_range=pred.max()-pred.min())


dataset=BraTSDataset(DATA_ROOT)
loader=DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=True)

netG=Generator().to(DEVICE)
optG=torch.optim.Adam(netG.parameters(),lr=LR)

L1=nn.L1Loss()
SSIM_L = lambda x,y: 1 - torch.mean(torch.tensor([
    compute_ssim(x[i,0].cpu().detach().numpy(), y[i,0].cpu().detach().numpy())
    for i in range(x.shape[0])
]).to(DEVICE))


for epoch in range(EPOCHS):

    netG.train()

    for imgs,ac,gt in loader:

        imgs=imgs.to(DEVICE)
        ac=ac.to(DEVICE)
        gt=gt.to(DEVICE)

        imgs=imgs.permute(1,0,2,3,4)
        gt=gt.permute(1,0,2,3,4)

        fake=netG(imgs,ac[0])

        loss_syn=sum((1-ac[0,i])*L1(fake[i],gt[i]) for i in range(4))
        loss_rec=sum(ac[0,i]*L1(fake[i],gt[i]) for i in range(4))
        loss_per=sum(perceptual_loss(fake[i],gt[i]) for i in range(4))

        loss_ssim = sum(SSIM_L(fake[i], gt[i]) for i in range(4))
        lossG = 50*loss_syn + 20*loss_rec + 20*loss_per + 10*loss_ssim


        optG.zero_grad()
        lossG.backward()
        optG.step()

    netG.eval()
    psnr_vals=[]
    ssim_vals=[]

    with torch.no_grad():

        for imgs,ac,gt in loader:

            imgs=imgs.to(DEVICE)
            ac=ac.to(DEVICE)
            gt=gt.to(DEVICE)

            imgs=imgs.permute(1,0,2,3,4)
            gt=gt.permute(1,0,2,3,4)

            fake=netG(imgs,ac[0])

            for i in range(4):
                if ac[0,i]==0:

                    pred=fake[i][0,0].cpu().numpy()
                    target=gt[i][0,0].cpu().numpy()

                    psnr_vals.append(compute_psnr(pred,target))
                    ssim_vals.append(compute_ssim(pred,target))

            save_sample(imgs, fake, ac[0], epoch)

            break

    print(f"Epoch {epoch} | PSNR {np.mean(psnr_vals):.2f} | SSIM {np.mean(ssim_vals):.4f}")



/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 221MB/s]  


Registering modalities...
Epoch 0 | PSNR 11.98 | SSIM 0.1368
Epoch 1 | PSNR 16.43 | SSIM 0.2300
Epoch 2 | PSNR 19.66 | SSIM 0.2521
Epoch 3 | PSNR 18.90 | SSIM 0.2707
Epoch 4 | PSNR 18.44 | SSIM 0.2978
Epoch 5 | PSNR 20.65 | SSIM 0.2618
Epoch 6 | PSNR 19.73 | SSIM 0.3784
Epoch 7 | PSNR 24.61 | SSIM 0.4688
Epoch 8 | PSNR 24.12 | SSIM 0.2283
Epoch 9 | PSNR 23.91 | SSIM 0.3726
Epoch 10 | PSNR 22.07 | SSIM 0.3558
Epoch 11 | PSNR 17.81 | SSIM 0.4537
Epoch 12 | PSNR 24.11 | SSIM 0.2757
Epoch 13 | PSNR 25.33 | SSIM 0.4792
Epoch 14 | PSNR 24.66 | SSIM 0.3575
Epoch 15 | PSNR 26.30 | SSIM 0.8123
Epoch 16 | PSNR 21.16 | SSIM 0.6771
Epoch 17 | PSNR 21.07 | SSIM 0.5090
Epoch 18 | PSNR 26.04 | SSIM 0.8644
Epoch 19 | PSNR 22.55 | SSIM 0.7544
Epoch 20 | PSNR 20.77 | SSIM 0.7347
Epoch 21 | PSNR 25.88 | SSIM 0.7970
Epoch 22 | PSNR 24.24 | SSIM 0.7292
Epoch 23 | PSNR 23.14 | SSIM 0.7597
Epoch 24 | PSNR 24.44 | SSIM 0.7045
Epoch 25 | PSNR 22.14 | SSIM 0.7624
Epoch 26 | PSNR 22.30 | SSIM 0.7771
Epoch 27 | P

In [4]:
import shutil

shutil.make_archive('/kaggle/working/synth_outputs', 'zip', SAVE_DIR)


'/kaggle/working/synth_outputs.zip'

In [2]:
# ================= IMPORTS =================
import os, random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ================= CONFIG =================
DATA_ROOT = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
SAVE_DIR  = "/kaggle/working/seg_outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4
EPOCHS = 40
LR = 1e-4
NUM_CASES  = 80     # reduce to 40 if slow
NUM_SLICES = 32

os.makedirs(SAVE_DIR, exist_ok=True)

# ================= UTILS =================

import cv2

def clean_mask(mask):
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask
    
def normalize(img):
    # MRI-safe min-max
    img = img - np.min(img)
    img = img / (np.max(img) + 1e-8)
    return img

def center_crop(img, size=192):
    h, w = img.shape
    s = size // 2
    return img[h//2-s:h//2+s, w//2-s:w//2+s]

def dice_coeff(pred, target, eps=1e-6):
    pred = (pred > 0.5).float()
    target = target.float()
    inter = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2*inter + eps) / (union + eps)
    return dice.mean().item()

def dice_loss(pred, target, eps=1e-6):
    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)
    inter = (pred * target).sum(dim=1)
    union = pred.sum(dim=1) + target.sum(dim=1)
    dice = (2*inter + eps) / (union + eps)
    return 1 - dice.mean()


def focal_dice_loss(pred, target):
    smooth = 1e-5

    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)

    intersection = (pred * target).sum(dim=1)
    dice = (2*intersection + smooth) / (pred.sum(dim=1) + target.sum(dim=1) + smooth)

    dice_loss_val = 1 - dice.mean()

    pt = pred*target + (1-pred)*(1-target)
    focal = ((1-pt)**2).mean()

    return dice_loss_val + 0.5*focal

# ================= DATASET =================
class BraTSSegDataset(Dataset):
    def __init__(self, root):
        self.data = []
        all_cases = []

        for folder in ["HGG", "LGG"]:
            path = os.path.join(root, folder)
            if os.path.exists(path):
                all_cases += [os.path.join(path, c) for c in os.listdir(path)]

        all_cases = all_cases[:NUM_CASES]

        for case in all_cases:
            files = os.listdir(case)

            t1=t2=t1ce=flair=seg=None
            for f in files:
                name=f.lower()
                if "_t1." in name and "ce" not in name: t1=f
                elif "_t2." in name: t2=f
                elif "t1ce" in name: t1ce=f
                elif "flair" in name: flair=f
                elif "seg" in name: seg=f

            if None in [t1,t2,t1ce,flair,seg]:
                continue

            vols=[]
            for fname in [t1,t2,t1ce,flair]:
                vol = nib.load(os.path.join(case,fname)).get_fdata(dtype=np.float32)
                z = vol.shape[2]
                start = (z-NUM_SLICES)//2
                vols.append(vol[:,:,start:start+NUM_SLICES])

            seg_vol = nib.load(os.path.join(case,seg)).get_fdata(dtype=np.float32)
            z = seg_vol.shape[2]
            start = (z-NUM_SLICES)//2
            seg_vol = seg_vol[:,:,start:start+NUM_SLICES]

            self.data.append((vols, seg_vol))

    def __len__(self):
        return len(self.data) * NUM_SLICES

    def __getitem__(self, idx):
        case_id = idx // NUM_SLICES
        slice_id = idx % NUM_SLICES

        vols, seg_vol = self.data[case_id]

        imgs=[]
        for vol in vols:
            img = center_crop(vol[:,:,slice_id])
            imgs.append(normalize(img))

        imgs = np.stack(imgs)             # (4,H,W)
        mask = center_crop(seg_vol[:,:,slice_id])
        mask = (mask > 0).astype(np.float32)  # WT binary

        imgs = torch.from_numpy(imgs).float()
        mask = torch.from_numpy(mask).unsqueeze(0).float()

        if mask.sum() == 0:
            if random.random() < 0.6:
                return self.__getitem__(random.randint(0, len(self)-1))

        return imgs, mask

# ================= MODEL =================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(4,32)
        self.enc2 = ConvBlock(32,64)
        self.enc3 = ConvBlock(64,128)
        self.enc4 = ConvBlock(128,256)
        self.pool = nn.MaxPool2d(2)

        self.up1 = nn.ConvTranspose2d(128,64,2,2)
        self.dec1 = ConvBlock(128,64)
        self.up2 = nn.ConvTranspose2d(64,32,2,2)
        self.dec2 = ConvBlock(64,32)

        self.out = nn.Conv2d(32,1,1)

    def forward(self,x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))

        x = self.up1(x3)
        x = self.dec1(torch.cat([x,x2],1))
        x = self.up2(x)
        x = self.dec2(torch.cat([x,x1],1))

        return torch.sigmoid(self.out(x))

# ================= TRAIN =================
dataset = BraTSSegDataset(DATA_ROOT)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = UNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
BCE = nn.BCELoss()

for epoch in range(EPOCHS):

    model.train()
    for imgs,mask in loader:
        imgs = imgs.to(DEVICE)
        mask = mask.to(DEVICE)

        pred = model(imgs)

        loss = focal_dice_loss(pred, mask)

        opt.zero_grad()
        loss.backward()
        opt.step()

    # ===== Evaluation =====
    model.eval()
    dices=[]

    with torch.no_grad():
        for imgs,mask in loader:
            imgs = imgs.to(DEVICE)
            mask = mask.to(DEVICE)

            pred = model(imgs)

            dices.append(dice_coeff(pred,mask))

            # save sample
            p = pred[0,0].cpu().numpy()
            p = (p > 0.5).astype(np.uint8)
            p = clean_mask(p)
            m = mask[0,0].cpu().numpy()
            i = imgs[0,0].cpu().numpy()

            fig,ax = plt.subplots(1,3,figsize=(9,3))
            ax[0].imshow(i,cmap="gray"); ax[0].set_title("Input")
            ax[1].imshow(m,cmap="gray"); ax[1].set_title("GT")
            ax[2].imshow(p,cmap="gray"); ax[2].set_title("Pred")
            for a in ax: a.axis("off")

            plt.savefig(f"{SAVE_DIR}/epoch{epoch}.png")
            plt.close()
            break

    print(f"Epoch {epoch} | Dice: {np.mean(dices):.4f}")

Epoch 0 | Dice: 0.5627
Epoch 1 | Dice: 0.7665
Epoch 2 | Dice: 0.7711
Epoch 3 | Dice: 0.9637
Epoch 4 | Dice: 0.8100
Epoch 5 | Dice: 0.7619
Epoch 6 | Dice: 0.6794
Epoch 7 | Dice: 0.9342
Epoch 8 | Dice: 0.8943
Epoch 9 | Dice: 0.9484
Epoch 10 | Dice: 0.9482
Epoch 11 | Dice: 0.9206
Epoch 12 | Dice: 0.9097
Epoch 13 | Dice: 0.9018
Epoch 14 | Dice: 0.7166
Epoch 15 | Dice: 0.9319
Epoch 16 | Dice: 0.9023
Epoch 17 | Dice: 0.4655
Epoch 18 | Dice: 0.9413
Epoch 19 | Dice: 0.9334
Epoch 20 | Dice: 0.8830
Epoch 21 | Dice: 0.8584
Epoch 22 | Dice: 0.8822
Epoch 23 | Dice: 0.7123
Epoch 24 | Dice: 0.9376
Epoch 25 | Dice: 0.9094
Epoch 26 | Dice: 0.6151
Epoch 27 | Dice: 0.6592
Epoch 28 | Dice: 0.9094
Epoch 29 | Dice: 0.9488
Epoch 30 | Dice: 0.9517
Epoch 31 | Dice: 0.9139
Epoch 32 | Dice: 0.9225
Epoch 33 | Dice: 0.8566
Epoch 34 | Dice: 0.6919
Epoch 35 | Dice: 0.9520
Epoch 36 | Dice: 0.9563
Epoch 37 | Dice: 0.9154
Epoch 38 | Dice: 0.9525
Epoch 39 | Dice: 0.9388


In [4]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATCH_SIZE = 64   # or 64 if still OOM

# =========================
# 📌 RANDOM CROP (PATCH)
# =========================
def random_crop(img, mask, size=128):
    _, H, W, D = img.shape

    h = np.random.randint(0, H - size)
    w = np.random.randint(0, W - size)
    d = np.random.randint(0, D - size)

    return (
        img[:, h:h+size, w:w+size, d:d+size],
        mask[h:h+size, w:w+size, d:d+size]
    )

# =========================
# 📌 DATASET
# =========================
class BraTSDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for subdir, _, files in os.walk(root_dir):
            if any("_flair.nii" in f for f in files):
                self.samples.append(subdir)

    def __len__(self):
        return len(self.samples)

    def normalize(self, img):
        if np.std(img) != 0:
            img = (img - np.mean(img)) / np.std(img)
        return img

    def __getitem__(self, idx):
        path = self.samples[idx]

        def load(name):
            file = [f for f in os.listdir(path) if name in f][0]
            return nib.load(os.path.join(path, file)).get_fdata()

        t1 = self.normalize(load("_t1.nii"))
        t1ce = self.normalize(load("_t1ce.nii"))
        t2 = self.normalize(load("_t2.nii"))
        flair = self.normalize(load("_flair.nii"))
        seg = load("_seg.nii")

        seg[seg == 4] = 3

        image = np.stack([t1, t1ce, t2, flair], axis=0)

        # 🔥 PATCH TRAINING
        image, seg = random_crop(image, seg, PATCH_SIZE)

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(seg, dtype=torch.long),
            path
        )

# =========================
# 📌 MODEL (LIGHTWEIGHT 3D U-NET)
# =========================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diffZ = x2.size()[2] - x1.size()[2]
        diffY = x2.size()[3] - x1.size()[3]
        diffX = x2.size()[4] - x1.size()[4]

        x1 = F.pad(x1, [diffX//2, diffX - diffX//2,
                        diffY//2, diffY - diffY//2,
                        diffZ//2, diffZ - diffZ//2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.inc = DoubleConv(4, 8)
        self.down1 = Down(8, 16)
        self.down2 = Down(16, 32)

        self.up1 = Up(32, 16)
        self.up2 = Up(16, 8)

        self.outc = nn.Conv3d(8, 4, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)

        x = self.up1(x3, x2)
        x = self.up2(x, x1)

        return self.outc(x)

# =========================
# 📌 LOSS
# =========================
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes=4).permute(0,4,1,2,3).float()

        intersection = torch.sum(probs * targets_onehot)
        union = torch.sum(probs) + torch.sum(targets_onehot)

        return 1 - (2 * intersection + 1e-5) / (union + 1e-5)

class SegLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice(logits, targets)

# =========================
# 📌 TRAIN (MIXED PRECISION)
# =========================

torch.cuda.empty_cache()

def train(model, loader, optimizer, loss_fn, epochs=5):
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for img, mask, _ in loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(img)
                loss = loss_fn(out, mask)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

# =========================
# 📌 SAVE PREDICTIONS
# =========================
def save_predictions(model, loader, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    with torch.no_grad():
        for img, mask, path in loader:
            img = img.to(DEVICE)

            out = model(img)
            pred = torch.argmax(out, dim=1).cpu().numpy()[0]
            mask = mask.numpy()[0]

            slice_idx = pred.shape[2] // 2

            fig, axs = plt.subplots(1, 3, figsize=(12, 4))

            axs[0].imshow(img.cpu().numpy()[0, 0, :, :, slice_idx], cmap="gray")
            axs[0].set_title("Input (T1)")

            axs[1].imshow(mask[:, :, slice_idx], cmap="jet")
            axs[1].set_title("Ground Truth")

            axs[2].imshow(pred[:, :, slice_idx], cmap="jet")
            axs[2].set_title("Prediction")

            for ax in axs:
                ax.axis("off")

            name = os.path.basename(path[0])
            save_path = os.path.join(output_dir, name + "_compare.png")

            plt.savefig(save_path, bbox_inches='tight')
            plt.close()

            print("Saved:", save_path)

# =========================
# 🚀 MAIN
# =========================
if __name__ == "__main__":

    dataset_path = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
    output_dir = "/kaggle/working/output"

    dataset = BraTSDataset(dataset_path)
    dataset.samples = dataset.samples[:50]
    loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)

    model = UNet3D().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = SegLoss()

    print("Training...")
    train(model, loader, optimizer, loss_fn, epochs=5)

    print("Saving predictions...")
    save_predictions(model, loader, output_dir)

    print("Done!")

Training...
Epoch 1, Loss: 2.1250
Epoch 2, Loss: 2.0186
Epoch 3, Loss: 1.9349
Epoch 4, Loss: 1.8678
Epoch 5, Loss: 1.8142
Saving predictions...
Saved: /kaggle/working/output/BraTS19_TCIA04_343_1_compare.png
Saved: /kaggle/working/output/BraTS19_2013_3_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_AYA_1_compare.png
Saved: /kaggle/working/output/BraTS19_TCIA01_412_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_AVT_1_compare.png
Saved: /kaggle/working/output/BraTS19_TCIA02_226_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_ATN_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_AQP_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_ARZ_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_ALU_1_compare.png
Saved: /kaggle/working/output/BraTS19_TCIA05_444_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_AVF_1_compare.png
Saved: /kaggle/working/output/BraTS19_TCIA02_321_1_compare.png
Saved: /kaggle/working/output/BraTS19_CBICA_BGN_

In [3]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATCH_SIZE = 128

# =========================
# 📌 RANDOM CROP (PATCH)
# =========================
def random_crop(img, mask, size=128):
    _, H, W, D = img.shape

    h = np.random.randint(0, H - size)
    w = np.random.randint(0, W - size)
    d = np.random.randint(0, D - size)

    return (
        img[:, h:h+size, w:w+size, d:d+size],
        mask[h:h+size, w:w+size, d:d+size]
    )

# =========================
# 📌 DATASET
# =========================
class BraTSDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for subdir, _, files in os.walk(root_dir):
            if any("_flair.nii" in f for f in files):
                self.samples.append(subdir)

    def __len__(self):
        return len(self.samples)

    def normalize(self, img):
        if np.std(img) != 0:
            img = (img - np.mean(img)) / np.std(img)
        return img

    def __getitem__(self, idx):
        path = self.samples[idx]

        def load(name):
            file = [f for f in os.listdir(path) if name in f][0]
            return nib.load(os.path.join(path, file)).get_fdata()

        t1 = self.normalize(load("_t1.nii"))
        t1ce = self.normalize(load("_t1ce.nii"))
        t2 = self.normalize(load("_t2.nii"))
        flair = self.normalize(load("_flair.nii"))
        seg = load("_seg.nii")

        seg[seg == 4] = 3

        image = np.stack([t1, t1ce, t2, flair], axis=0)

        # 🔥 PATCH TRAINING
        image, seg = random_crop(image, seg, PATCH_SIZE)

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(seg, dtype=torch.long),
            path
        )

# =========================
# 📌 MODEL (LIGHTWEIGHT 3D U-NET)
# =========================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diffZ = x2.size()[2] - x1.size()[2]
        diffY = x2.size()[3] - x1.size()[3]
        diffX = x2.size()[4] - x1.size()[4]

        x1 = F.pad(x1, [diffX//2, diffX - diffX//2,
                        diffY//2, diffY - diffY//2,
                        diffZ//2, diffZ - diffZ//2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()
        self.inc = DoubleConv(4, 16)
        self.down1 = Down(16, 32)
        self.down2 = Down(32, 64)
        self.down3 = Down(64, 128)

        self.up1 = Up(128, 64)
        self.up2 = Up(64, 32)
        self.up3 = Up(32, 16)

        self.outc = nn.Conv3d(16, 4, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)

        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)

        return self.outc(x)

# =========================
# 📌 LOSS
# =========================
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes=4).permute(0,4,1,2,3).float()

        intersection = torch.sum(probs * targets_onehot)
        union = torch.sum(probs) + torch.sum(targets_onehot)

        return 1 - (2 * intersection + 1e-5) / (union + 1e-5)

class SegLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice(logits, targets)

# =========================
# 📌 TRAIN (MIXED PRECISION)
# =========================
def train(model, loader, optimizer, loss_fn, epochs=5):
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for img, mask, _ in loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(img)
                loss = loss_fn(out, mask)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

# =========================
# 📌 SAVE PREDICTIONS
# =========================
import matplotlib.pyplot as plt

def save_predictions(model, loader, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    with torch.no_grad():
        for img, mask, path in loader:
            img = img.to(DEVICE)

            out = model(img)
            pred = torch.argmax(out, dim=1).cpu().numpy()[0]
            mask = mask.numpy()[0]

            slice_idx = pred.shape[2] // 2

            fig, axs = plt.subplots(1, 3, figsize=(12, 4))

            axs[0].imshow(img.cpu().numpy()[0, 0, :, :, slice_idx], cmap="gray")
            axs[0].set_title("Input (T1)")

            axs[1].imshow(mask[:, :, slice_idx], cmap="jet")
            axs[1].set_title("Ground Truth")

            axs[2].imshow(pred[:, :, slice_idx], cmap="jet")
            axs[2].set_title("Prediction")

            for ax in axs:
                ax.axis("off")

            name = os.path.basename(path[0])
            save_path = os.path.join(output_dir, name + "_compare.png")

            plt.savefig(save_path, bbox_inches='tight')
            plt.close()

            print("Saved:", save_path)

# =========================
# 🚀 MAIN
# =========================
if __name__ == "__main__":

    dataset_path = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
    output_dir = "/kaggle/working/output2"

    dataset = BraTSDataset(dataset_path)
    loader = DataLoader(dataset, batch_size=1, shuffle=True)

    model = UNet3D().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = SegLoss()

    print("Training...")
    train(model, loader, optimizer, loss_fn, epochs=5)

    print("Saving predictions...")
    save_predictions(model, loader, output_dir)

    print("Done!")

Training...
Epoch 1, Loss: 1.4985


KeyboardInterrupt: 

In [5]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATCH_SIZE = 48

# =========================
# 🔥 TUMOR-FOCUSED CROP
# =========================
def tumor_crop(img, mask, size=48):
    _, H, W, D = img.shape

    coords = np.argwhere(mask > 0)

    if len(coords) > 0:
        center = coords[np.random.choice(len(coords))]
        h, w, d = center

        h = np.clip(h - size//2, 0, H - size)
        w = np.clip(w - size//2, 0, W - size)
        d = np.clip(d - size//2, 0, D - size)
    else:
        h = np.random.randint(0, H - size)
        w = np.random.randint(0, W - size)
        d = np.random.randint(0, D - size)

    return (
        img[:, h:h+size, w:w+size, d:d+size],
        mask[h:h+size, w:w+size, d:d+size]
    )

# =========================
# 📌 DATASET
# =========================
class BraTSDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []

        for subdir, _, files in os.walk(root_dir):
            if any("_flair.nii" in f for f in files):
                self.samples.append(subdir)

    def __len__(self):
        return len(self.samples)

    def normalize(self, img):
        if np.std(img) != 0:
            img = (img - np.mean(img)) / np.std(img)
        return img

    def __getitem__(self, idx):
        path = self.samples[idx]

        def load(name):
            file = [f for f in os.listdir(path) if name in f][0]
            return nib.load(os.path.join(path, file)).get_fdata()

        t1 = self.normalize(load("_t1.nii"))
        t1ce = self.normalize(load("_t1ce.nii"))
        t2 = self.normalize(load("_t2.nii"))
        flair = self.normalize(load("_flair.nii"))
        seg = load("_seg.nii")

        seg[seg == 4] = 3

        image = np.stack([t1, t1ce, t2, flair], axis=0)

        # 🔥 MIXED SAMPLING (BEST)
        if np.random.rand() > 0.5:
            image, seg = tumor_crop(image, seg, PATCH_SIZE)
        else:
            image, seg = tumor_crop(image, seg, PATCH_SIZE)

        return (
            torch.tensor(image, dtype=torch.float32),
            torch.tensor(seg, dtype=torch.long),
            path
        )

# =========================
# 📌 MODEL (SAFE VERSION)
# =========================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool3d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose3d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)

        diffZ = x2.size()[2] - x1.size()[2]
        diffY = x2.size()[3] - x1.size()[3]
        diffX = x2.size()[4] - x1.size()[4]

        x1 = F.pad(x1, [diffX//2, diffX - diffX//2,
                        diffY//2, diffY - diffY//2,
                        diffZ//2, diffZ - diffZ//2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.inc = DoubleConv(4, 8)
        self.down1 = Down(8, 16)
        self.down2 = Down(16, 32)

        self.up1 = Up(32, 16)
        self.up2 = Up(16, 8)

        self.outc = nn.Conv3d(8, 4, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)

        x = self.up1(x3, x2)
        x = self.up2(x, x1)

        return self.outc(x)

# =========================
# 📌 LOSS
# =========================
class DiceLoss(nn.Module):
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, num_classes=4).permute(0,4,1,2,3).float()

        intersection = torch.sum(probs * targets_onehot)
        union = torch.sum(probs) + torch.sum(targets_onehot)

        return 1 - (2 * intersection + 1e-5) / (union + 1e-5)

class SegLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
        self.dice = DiceLoss()

    def forward(self, logits, targets):
        return self.ce(logits, targets) + self.dice(logits, targets)

# =========================
# 📌 TRAIN
# =========================
def train(model, loader, optimizer, loss_fn, epochs=5):
    scaler = torch.amp.GradScaler("cuda")

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for img, mask, _ in loader:
            img, mask = img.to(DEVICE), mask.to(DEVICE)

            optimizer.zero_grad()

            with torch.amp.autocast("cuda"):
                out = model(img)
                loss = loss_fn(out, mask)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

# =========================
# 📌 SAVE PNG OUTPUT
# =========================
def save_predictions(model, loader, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    model.eval()

    with torch.no_grad():
        for img, mask, path in loader:
            img = img.to(DEVICE)

            out = model(img)
            pred = torch.argmax(out, dim=1).cpu().numpy()[0]
            mask = mask.numpy()[0]

            slice_idx = pred.shape[2] // 2

            fig, axs = plt.subplots(1, 3, figsize=(12, 4))

            axs[0].imshow(img.cpu().numpy()[0, 0, :, :, slice_idx], cmap="gray")
            axs[0].set_title("Input")

            axs[1].imshow(mask[:, :, slice_idx], cmap="jet")
            axs[1].set_title("GT")

            axs[2].imshow(pred[:, :, slice_idx], cmap="jet")
            axs[2].set_title("Pred")

            for ax in axs:
                ax.axis("off")

            name = os.path.basename(path[0])
            save_path = os.path.join(output_dir, name + ".png")

            plt.savefig(save_path, bbox_inches='tight')
            plt.close()

            print("Saved:", save_path)

# =========================
# 🚀 MAIN
# =========================
if __name__ == "__main__":

    dataset_path = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
    output_dir = "/kaggle/working/output1"

    dataset = BraTSDataset(dataset_path)

    # 🔥 LIMIT DATA (FOR KAGGLE)
    dataset.samples = dataset.samples[:50]

    loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=0)

    model = UNet3D().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loss_fn = SegLoss()

    torch.cuda.empty_cache()

    print("Training...")
    train(model, loader, optimizer, loss_fn, epochs=5)

    print("Saving predictions...")
    save_predictions(model, loader, output_dir)

    print("Done!")

Training...
Epoch 1, Loss: 1.9671
Epoch 2, Loss: 1.7341
Epoch 3, Loss: 1.6559
Epoch 4, Loss: 1.6034
Epoch 5, Loss: 1.5674
Saving predictions...
Saved: /kaggle/working/output1/BraTS19_2013_27_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_AYG_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA03_265_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA02_314_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA06_372_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA04_343_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_APR_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_AXW_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA08_242_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_AWG_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA03_138_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_AQP_1.png
Saved: /kaggle/working/output1/BraTS19_TCIA03_338_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_ASA_1.png
Saved: /kaggle/working/output1/BraTS19_CBICA_ASV_1.png
Saved: /kaggle/working/out

In [6]:
# ================= IMPORTS =================
import os, random
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# ================= CONFIG =================
DATA_ROOT = "/kaggle/input/datasets/aryashah2k/brain-tumor-segmentation-brats-2019/MICCAI_BraTS_2019_Data_Training"
SAVE_DIR  = "/kaggle/working/seg_outputs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 4
EPOCHS = 40
LR = 1e-4
NUM_CASES  = 80
NUM_SLICES = 32

os.makedirs(SAVE_DIR, exist_ok=True)

# ================= UTILS =================
def normalize(img):
    img = img - np.min(img)
    img = img / (np.max(img) + 1e-8)
    return img

def center_crop(img, size=192):
    h, w = img.shape
    s = size // 2
    return img[h//2-s:h//2+s, w//2-s:w//2+s]

# ================= DATASET =================
class BraTSSegDataset(Dataset):
    def __init__(self, root):
        self.data = []
        all_cases = []

        for folder in ["HGG", "LGG"]:
            path = os.path.join(root, folder)
            if os.path.exists(path):
                all_cases += [os.path.join(path, c) for c in os.listdir(path)]

        all_cases = all_cases[:NUM_CASES]

        for case in all_cases:
            files = os.listdir(case)

            t1=t2=t1ce=flair=seg=None
            for f in files:
                name=f.lower()
                if "_t1." in name and "ce" not in name: t1=f
                elif "_t2." in name: t2=f
                elif "t1ce" in name: t1ce=f
                elif "flair" in name: flair=f
                elif "seg" in name: seg=f

            if None in [t1,t2,t1ce,flair,seg]:
                continue

            vols=[]
            for fname in [t1,t2,t1ce,flair]:
                vol = nib.load(os.path.join(case,fname)).get_fdata(dtype=np.float32)
                z = vol.shape[2]
                start = (z-NUM_SLICES)//2
                vols.append(vol[:,:,start:start+NUM_SLICES])

            seg_vol = nib.load(os.path.join(case,seg)).get_fdata(dtype=np.float32)
            seg_vol[seg_vol == 4] = 3   # 🔥 FIX LABEL

            z = seg_vol.shape[2]
            start = (z-NUM_SLICES)//2
            seg_vol = seg_vol[:,:,start:start+NUM_SLICES]

            self.data.append((vols, seg_vol))

    def __len__(self):
        return len(self.data) * NUM_SLICES

    def __getitem__(self, idx):
        case_id = idx // NUM_SLICES
        slice_id = idx % NUM_SLICES

        vols, seg_vol = self.data[case_id]

        imgs=[]
        for vol in vols:
            img = center_crop(vol[:,:,slice_id])
            imgs.append(normalize(img))

        imgs = np.stack(imgs)   # (4,H,W)
        mask = center_crop(seg_vol[:,:,slice_id])

        imgs = torch.from_numpy(imgs).float()
        mask = torch.from_numpy(mask).long()   # 🔥 MULTI-CLASS

        return imgs, mask

# ================= MODEL =================
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(4,32)
        self.enc2 = ConvBlock(32,64)
        self.enc3 = ConvBlock(64,128)
        self.pool = nn.MaxPool2d(2)

        self.up1 = nn.ConvTranspose2d(128,64,2,2)
        self.dec1 = ConvBlock(128,64)
        self.up2 = nn.ConvTranspose2d(64,32,2,2)
        self.dec2 = ConvBlock(64,32)

        self.out = nn.Conv2d(32,4,1)   # 🔥 MULTI-CLASS

    def forward(self,x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))

        x = self.up1(x3)
        x = self.dec1(torch.cat([x,x2],1))
        x = self.up2(x)
        x = self.dec2(torch.cat([x,x1],1))

        return self.out(x)   # NO sigmoid

# ================= LOSS =================
CE = nn.CrossEntropyLoss()

def multiclass_dice_loss(pred, target, num_classes=4):
    pred = torch.softmax(pred, dim=1)
    target_onehot = F.one_hot(target, num_classes).permute(0,3,1,2).float()

    intersection = (pred * target_onehot).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target_onehot.sum(dim=(2,3))

    dice = (2*intersection + 1e-5) / (union + 1e-5)
    return 1 - dice.mean()

# ================= TRAIN =================
dataset = BraTSSegDataset(DATA_ROOT)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model = UNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):

    model.train()
    for imgs,mask in loader:
        imgs = imgs.to(DEVICE)
        mask = mask.to(DEVICE)

        pred = model(imgs)

        loss = CE(pred, mask) + multiclass_dice_loss(pred, mask)

        opt.zero_grad()
        loss.backward()
        opt.step()

    # ===== Evaluation =====
    model.eval()

    with torch.no_grad():
        for imgs,mask in loader:
            imgs = imgs.to(DEVICE)
            mask = mask.to(DEVICE)

            pred = model(imgs)
            pred_cls = torch.argmax(pred, dim=1)

            # visualize
            p = pred_cls[0].cpu().numpy()
            m = mask[0].cpu().numpy()
            i = imgs[0,0].cpu().numpy()

            fig,ax = plt.subplots(1,3,figsize=(12,4))
            ax[0].imshow(i,cmap="gray"); ax[0].set_title("Input")
            ax[1].imshow(m,cmap="jet"); ax[1].set_title("GT")
            ax[2].imshow(p,cmap="jet"); ax[2].set_title("Pred")

            for a in ax: a.axis("off")

            plt.savefig(f"{SAVE_DIR}/epoch_{epoch}.png")
            plt.close()
            break

    print(f"Epoch {epoch} done")

Epoch 0 done
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done
Epoch 8 done
Epoch 9 done
Epoch 10 done
Epoch 11 done
Epoch 12 done
Epoch 13 done
Epoch 14 done
Epoch 15 done
Epoch 16 done
Epoch 17 done
Epoch 18 done
Epoch 19 done
Epoch 20 done
Epoch 21 done
Epoch 22 done
Epoch 23 done
Epoch 24 done
Epoch 25 done
Epoch 26 done
Epoch 27 done
Epoch 28 done
Epoch 29 done
Epoch 30 done
Epoch 31 done
Epoch 32 done
Epoch 33 done
Epoch 34 done
Epoch 35 done
Epoch 36 done
Epoch 37 done
Epoch 38 done
Epoch 39 done


In [7]:
import torch

# =========================
# 🧠 BASIC DICE FUNCTION
# =========================
def dice_score(pred, target, eps=1e-5):
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    return (2 * intersection + eps) / (union + eps)


# =========================
# 🔥 MULTI-CLASS DICE (BRATS)
# =========================
def compute_brats_dice(pred, target):
    """
    pred, target: numpy or torch tensors of shape (H, W)
    values: 0,1,2,3
    """

    if isinstance(pred, torch.Tensor):
        pred = pred.cpu()
    if isinstance(target, torch.Tensor):
        target = target.cpu()

    # Whole Tumor (WT): labels 1,2,3
    pred_wt = (pred > 0).float()
    target_wt = (target > 0).float()

    # Tumor Core (TC): labels 1,3
    pred_tc = ((pred == 1) | (pred == 3)).float()
    target_tc = ((target == 1) | (target == 3)).float()

    # Enhancing Tumor (ET): label 3
    pred_et = (pred == 3).float()
    target_et = (target == 3).float()

    dice_wt = dice_score(pred_wt, target_wt)
    dice_tc = dice_score(pred_tc, target_tc)
    dice_et = dice_score(pred_et, target_et)

    return {
        "WT": dice_wt.item(),
        "TC": dice_tc.item(),
        "ET": dice_et.item()
    }

In [9]:
model.eval()

total_wt, total_tc, total_et = 0, 0, 0
count = 0

with torch.no_grad():
    for imgs, mask in loader:

        imgs = imgs.to(DEVICE)
        mask = mask.to(DEVICE)

        # 🔥 THIS LINE WAS MISSING
        output = model(imgs)

        pred = torch.argmax(output, dim=1)

        scores = compute_brats_dice(pred[0], mask[0])

        total_wt += scores["WT"]
        total_tc += scores["TC"]
        total_et += scores["ET"]
        count += 1

print("Average Dice:")
print("WT:", total_wt / count)
print("TC:", total_tc / count)
print("ET:", total_et / count)

Average Dice:
WT: 0.8854083571483888
TC: 0.9126755346210288
ET: 0.8850399062787316
